In [0]:
# ingestion notebook

# Spark
from pyspark.sql.functions import *

# API
import requests
import json
from typing import Dict, Any, List, Optional

# Logging
import logging

In [0]:
# basic logging

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

logger = logging.getLogger(__name__)


In [0]:
# dynamic API parameters

API_CONFIG = {
    "base_url": "https://meowfacts.herokuapp.com/",
    "headers": None,
    "params": None,
    "pagination": False,
    "data_key": None
}

In [0]:
# function API loader

def get_api_data(config: dict):  
    url = config.get("base_url")
    headers = config.get("headers")
    params = config.get("params")
    pagination = config.get("pagination")
    data_key = config.get("data_key")

    logger.info(f"Calling API: {url}")

    response = requests.get(
        url,
        headers=headers,
        params=params
    )

    try:
        result = response.json()
    except Exception as e:
        return {"error": f"JSON decode failed: {str(e)}", "status_code": response.status_code, "text": response.text}

    if response.status_code != 200:
        return {"error": response.status_code}
    else:
        return result

In [0]:
# convert json API data to key-value records

def json_to_df(api_data: dict) -> list:

    if not api_data:
        raise ValueError("No data received from API")

    records = []
    for key, value in api_data.items():
        if isinstance(value, list):
            for item in value:
                records.append({
                    "key": key,
                    "value": str(item)
                })
        elif isinstance(value, dict):
            records.append({
                "key": key,
                "value": json.dumps(value)
            })
        else:
            records.append({
                "key": key,
                "value": str(value)
            })
    return spark.createDataFrame(records)
# convert API data to Spark DataFrame)



    # return spark.createDataFrame(api_data)


In [0]:
def add_metadata(df):
    return df.withColumn("ingested_at", current_timestamp())

In [0]:
api_data = get_api_data(API_CONFIG)

key_value_data = json_to_df(api_data)

key_value_data = add_metadata(key_value_data)

key_value_data.show()

# df = json_to_df(api_data)

# display(df)